# Notebook 02: Input Pipeline & Baseline CNN from Scratch

**Objectives:**
1. Build a high-performance `tf.data` input pipeline (Resizing, Batching, Caching, Prefetching).
2. Implement data augmentation using Keras layers.
3. Build a 4-block Convolutional Neural Network from scratch.
4. Explain all layers, parameter counting formulas, activation functions, loss, and optimizer.
5. Train with `EarlyStopping` on free Colab T4 GPU (Runtime: ~2 minutes).
6. Plot training/validation curves and analyze overfitting.

In [ ]:
# Cell 0: Setup Directories & Colab Dependencies
import os
!pip install -q importlib_resources tensorflow-datasets --upgrade

RESULTS_DIR = 'results' if os.path.exists('results') or not os.path.exists('../results') else '../results'
MODELS_DIR = 'models' if os.path.exists('models') or not os.path.exists('../models') else '../models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Results: {os.path.abspath(RESULTS_DIR)}")
print(f"Models:  {os.path.abspath(MODELS_DIR)}")

In [ ]:
# Cell 1: Environment & Seed Fixing
import random
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Cell 2: Load Stratified Splits
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds_raw, ds_info = tfds.load(
    'plant_village',
    split='train[:70%]',
    as_supervised=True,
    with_info=True
)
val_ds_raw = tfds.load('plant_village', split='train[70%:85%]', as_supervised=True)
test_ds_raw = tfds.load('plant_village', split='train[85%:]', as_supervised=True)

class_names = ds_info.features['label'].names
num_classes = ds_info.features['label'].num_classes
print(f"Loaded {num_classes} classes successfully.")

In [ ]:
# Cell 3: Data Augmentation Pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    tf.keras.layers.RandomRotation(0.1, seed=SEED),
    tf.keras.layers.RandomZoom(0.1, seed=SEED),
    tf.keras.layers.RandomContrast(0.1, seed=SEED),
], name="data_augmentation")

# Demonstrate augmentation on a single sample image
for sample_img, sample_lbl in train_ds_raw.take(1):
    sample_img_resized = tf.image.resize(sample_img, IMG_SIZE)
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 4, 1)
    plt.imshow(sample_img_resized.numpy().astype("uint8"))
    plt.title("Original (224x224)")
    plt.axis("off")
    
    for i in range(3):
        augmented = data_augmentation(tf.expand_dims(sample_img_resized, 0))
        plt.subplot(1, 4, i + 2)
        plt.imshow(augmented[0].numpy().astype("uint8"))
        plt.title(f"Augmented #{i+1}")
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'augmented_samples.png'))
    plt.show()

In [ ]:
# Cell 4: High-Performance tf.data Input Pipeline
def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0  # normalize to [0, 1]
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = val_ds_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = test_ds_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Pipelines configured successfully without RAM bloat.")

In [ ]:
# Cell 5: Build Small Custom CNN Architecture from Scratch
from tensorflow.keras import layers, models

def build_baseline_cnn(input_shape=(224, 224, 3), num_classes=38):
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    
    # Block 1
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1')(x)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    
    # Block 2
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2')(x)
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    
    # Block 3
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3')(x)
    x = layers.MaxPooling2D((2, 2), name='pool3')(x)
    
    # Block 4
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu', name='conv4')(x)
    x = layers.MaxPooling2D((2, 2), name='pool4')(x)
    
    # Classification Head
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.3, seed=SEED, name='dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='baseline_small_cnn')
    return model

baseline_model = build_baseline_cnn(num_classes=num_classes)
baseline_model.summary()

In [ ]:
# Cell 6: Compile and Train Baseline Model
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_path = os.path.join(MODELS_DIR, 'baseline_best.keras')
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

EPOCHS = 8
print("Starting baseline training...")
history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# Cell 7: Plot Training vs Validation Curves
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
plt.title('Baseline CNN: Accuracy vs Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss', marker='s')
plt.title('Baseline CNN: Loss vs Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (Sparse Cross Entropy)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'baseline_training_curves.png'), dpi=150)
plt.show()